# Task B Iteration v1 — Chroma + BGE Retrieval + Optional Cross-Encoder Rerank

Goal: improve high-recall candidate generation before API/LLM reranking.

This notebook uses the filtered Task A iteration data:

```text
data/processed/task_a_iteration_v0/train.parquet
data/processed/task_a_iteration_v0/internal_val.parquet
data/processed/task_a_iteration_v0/items.parquet
```

Pipeline tested here:

```text
Item metadata → BAAI/bge-small-en-v1.5 embeddings → ChromaDB
User train history → profile query + anchor item queries
Profile/anchor Chroma retrieval + popularity + gateway + category popularity
Union/deduplicate → top 500–1000 candidates
Evaluate Hit@K / Recall@K / NDCG@K
Optional: cross-encoder/ms-marco-MiniLM-L6-v2 rerank
```


## 0. Install dependencies
Run this once if your environment does not already have these packages.

In [1]:
# Uncomment if needed
%pip install -q sentence-transformers chromadb tqdm scikit-learn pyarrow pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install "torch>=2.4" torchvision torchaudio

ERROR: Could not find a version that satisfies the requirement torch>=2.4 (from versions: 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for torch>=2.4
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip uninstall -y sentence-transformers transformers tokenizers torch torchvision torchaudio

Found existing installation: sentence-transformers 5.5.0
Uninstalling sentence-transformers-5.5.0:
  Successfully uninstalled sentence-transformers-5.5.0
Found existing installation: transformers 5.8.1
Uninstalling transformers-5.8.1:
  Successfully uninstalled transformers-5.8.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install "torch==2.2.2" "torchvision==0.17.2" "torchaudio==2.2.2"
%pip install "transformers==4.41.2" "sentence-transformers==2.7.0" "tokenizers==0.19.1"
%pip install chromadb tqdm pandas numpy scikit-learn

  Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl.metadata (25 kB)
  Using cached torchvision-0.17.2-cp311-cp311-macosx_10_13_x86_64.whl.metadata (6.6 kB)
Using cached torch-2.2.2-cp311-none-macosx_10_9_x86_64.whl (150.8 MB)
Using cached torchvision-0.17.2-cp311-cp311-macosx_10_13_x86_64.whl (1.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 15.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]3 [torchaudio]]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 17.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.7 MB/s  0:00:00m eta 0:00:01
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub


In [1]:
import torch
import transformers
import sentence_transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sentence_transformers:", sentence_transformers.__version__)

from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

print("Loaded models successfully")

torch: 2.2.2
transformers: 4.41.2
sentence_transformers: 2.7.0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded models successfully


## 1. Imports and config

In [2]:
from pathlib import Path
import json
import math
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

TASK_A_DIR = Path("data/processed/task_a_iteration_v0")
CHROMA_DIR = Path("data/processed/task_b_chroma_bge_small")
CHROMA_COLLECTION = "amazon_items_bge_small_v1"

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

# Retrieval sizes: tune these while debugging recall.
N_PROFILE = 700
N_ANCHOR_TOTAL = 700
N_POPULAR = 300
N_GATEWAY = 300
N_CATEGORY_POPULAR = 300
FINAL_CANDIDATE_K = 1000

EVAL_KS = (10, 30, 100, 500, 1000)
RANDOM_STATE = 42

# Set True if you changed item_text/card construction and want to rebuild Chroma.
RESET_CHROMA_INDEX = False

## 2. Load filtered data

In [5]:
train_df = pd.read_parquet(TASK_A_DIR / "train.parquet")
internal_val_df = pd.read_parquet(TASK_A_DIR / "internal_val.parquet")
items_df = pd.read_parquet(TASK_A_DIR / "items.parquet")

print("Loaded:")
print("train:", train_df.shape)
print("internal_val:", internal_val_df.shape)
print("items:", items_df.shape)

for df in [train_df, internal_val_df, items_df]:
    df["domain"] = df["domain"].astype(str)
    df["parent_asin"] = df["parent_asin"].astype(str)

items_df = items_df.drop_duplicates(["domain", "parent_asin"]).reset_index(drop=True)

items_df["item_key"] = items_df["domain"] + "||" + items_df["parent_asin"]
train_df["item_key"] = train_df["domain"] + "||" + train_df["parent_asin"]
internal_val_df["item_key"] = internal_val_df["domain"] + "||" + internal_val_df["parent_asin"]

# item_lookup = {(row.domain, row.parent_asin): row.to_dict() for row in items_df.itertuples(index=False)}
item_lookup = {
    (str(r.domain), str(r.parent_asin)): r._asdict()
    for r in items_df.fillna("").itertuples(index=False)
}
items_df.head()

Loaded:
train: (190457, 13)
internal_val: (20408, 13)
items: (50927, 11)


,domain,parent_asin,main_category,title,store,categories,features,description,price,has_useful_metadata,n_train_reviews_for_item,item_key
0,Books,0316185361,Books,Service: A Navy SEAL at War,"Marcus Luttrell (Author), James D. Hornfischer",Books Biographies & Memoirs Leaders & Notable ...,"Marcus Luttrell, author of the #1 bestseller L...","Review Praise for SERVICE""An action-packed...r...",17.17,True,4,Books||0316185361
1,Books,1932225323,Books,Four Centuries of American Education,David Barton (Author),Books Education & Teaching Schools & Teaching,"For four centuries, religion, morality, and kn...",About the Author David Barton is the founder o...,6.99,True,3,Books||1932225323
2,Books,0062279068,Books,"Clark the Shark: Tooth Trouble, No. 1","Bruce Hale (Author), Guy Francis (Illustrator)",Books Children's Books Growing Up & Facts of Life,"Don’t shed a tear, 'cause there’s nothing to f...",Review “Francis’s bubbly illustrations of over...,4.99,True,3,Books||0062279068
3,Books,0937295760,Books,Kirsten: An American Girl : 1854 (The American...,Janet Beeler Shaw (Author),Books Boxed Sets Children's Books,Includes six paperback books in a cardboard sl...,,159.90,True,3,Books||0937295760
4,Books,0385529813,Books,Meat Eater: Adventures from the Life of an Ame...,Steven Rinella (Author),"Books Cookbooks, Food & Wine Cooking by Ingred...","“Revelatory . . . With every chapter, you get ...","Review “Truth be told, I have lived a life ple...",49.00,True,4,Books||0385529813


## 3. Build item text, item stats, popularity, gateway, and category tables

In [6]:
TEXT_COLS = ["title", "main_category", "categories", "features", "description", "store"]
for col in TEXT_COLS:
    if col not in items_df.columns:
        items_df[col] = ""
    items_df[col] = items_df[col].fillna("").astype(str)


def compact_text(x, max_chars=700):
    x = " ".join(str(x).split())
    return x[:max_chars]


def build_item_text(row):
    parts = []
    title = compact_text(row.get("title", ""), 180)
    if title:
        parts.append(f"Title: {title}")
    domain = compact_text(row.get("domain", ""), 80)
    if domain:
        parts.append(f"Domain: {domain}")
    main_category = compact_text(row.get("main_category", ""), 160)
    if main_category:
        parts.append(f"Main category: {main_category}")
    categories = compact_text(row.get("categories", ""), 350)
    if categories:
        parts.append(f"Categories: {categories}")
    store = compact_text(row.get("store", ""), 120)
    if store:
        parts.append(f"Store/brand: {store}")
    features = compact_text(row.get("features", ""), 500)
    if features:
        parts.append(f"Features: {features}")
    description = compact_text(row.get("description", ""), 700)
    if description:
        parts.append(f"Description: {description}")
    return "\n".join(parts)

items_df["item_text"] = items_df.apply(lambda r: build_item_text(r.to_dict()), axis=1)
items_df["item_card"] = items_df["item_text"].str.slice(0, 1200)

# Item stats from train only
item_stats = (
    train_df.groupby(["domain", "parent_asin", "item_key"])
    .agg(
        item_train_count=("rating", "size"),
        item_train_mean_rating=("rating", "mean"),
        item_train_std_rating=("rating", "std"),
    )
    .reset_index()
)

global_mean = float(train_df["rating"].mean())
reg = 10
item_stats["bayes_rating"] = (
    (item_stats["item_train_mean_rating"] * item_stats["item_train_count"] + global_mean * reg)
    / (item_stats["item_train_count"] + reg)
)
item_stats["popularity_score_raw"] = item_stats["bayes_rating"] * np.log1p(item_stats["item_train_count"])
item_stats["popularity_score"] = item_stats["popularity_score_raw"] / item_stats["popularity_score_raw"].max()

# Gateway score: items liked in users' first 3 train reviews
early = train_df.sort_values(["user_id", "timestamp", "domain", "parent_asin"]).copy()
early["user_rn_train"] = early.groupby("user_id").cumcount() + 1
early = early[early["user_rn_train"] <= 3]

gateway = (
    early.groupby(["domain", "parent_asin", "item_key"])
    .agg(
        early_count=("rating", "size"),
        early_mean_rating=("rating", "mean"),
    )
    .reset_index()
)
gateway["gateway_score_raw"] = gateway["early_mean_rating"] * np.log1p(gateway["early_count"])
if len(gateway) and gateway["gateway_score_raw"].max() > 0:
    gateway["gateway_score"] = gateway["gateway_score_raw"] / gateway["gateway_score_raw"].max()
else:
    gateway["gateway_score"] = 0.0

items_df = (
    items_df
    .merge(item_stats[["domain", "parent_asin", "item_train_count", "item_train_mean_rating", "popularity_score"]],
           on=["domain", "parent_asin"], how="left")
    .merge(gateway[["domain", "parent_asin", "early_count", "early_mean_rating", "gateway_score"]],
           on=["domain", "parent_asin"], how="left")
)

for col in ["item_train_count", "early_count"]:
    items_df[col] = items_df[col].fillna(0).astype(int)
for col in ["item_train_mean_rating", "popularity_score", "early_mean_rating", "gateway_score"]:
    items_df[col] = items_df[col].fillna(0.0).astype(float)

# Simple category key for category-aware popularity
items_df["category_key"] = (
    items_df["categories"].fillna("").astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.split(",")
    .str[0]
    .fillna("")
    .str.strip()
)
items_df.loc[items_df["category_key"].eq(""), "category_key"] = items_df["main_category"].fillna("").astype(str)

items_df[["domain", "parent_asin", "title", "item_train_count", "popularity_score", "gateway_score", "category_key"]].head()

,domain,parent_asin,title,item_train_count,popularity_score,gateway_score,category_key
0,Books,0316185361,Service: A Navy SEAL at War,2,0.194798,0.0,Books Biographies & Memoirs Leaders & Notable ...
1,Books,1932225323,Four Centuries of American Education,0,0.000000,0.0,Books Education & Teaching Schools & Teaching
2,Books,0062279068,"Clark the Shark: Tooth Trouble, No. 1",1,0.126358,0.0,Books Childrens Books Growing Up & Facts of Life
3,Books,0937295760,Kirsten: An American Girl : 1854 (The American...,0,0.000000,0.0,Books Boxed Sets Childrens Books
4,Books,0385529813,Meat Eater: Adventures from the Life of an Ame...,4,0.286853,0.0,Books Cookbooks


## 4. Build or load ChromaDB item index with BGE embeddings

This indexes item metadata once. If the collection already exists with the expected number of items, the notebook skips rebuilding.

In [ ]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("Loading embedding model:", EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_CHROMA_INDEX:
    try:
        client.delete_collection(CHROMA_COLLECTION)
        print("Deleted existing collection.")
    except Exception as e:
        print("No existing collection to delete:", e)

collection = client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

expected_n = len(items_df)
current_n = collection.count()
print("Current Chroma count:", current_n, "Expected:", expected_n)

if current_n != expected_n:
    print("Building/rebuilding Chroma collection...")
    try:
        client.delete_collection(CHROMA_COLLECTION)
    except Exception:
        pass
    collection = client.get_or_create_collection(
        name=CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"},
    )

    batch_size = 256
    for start in tqdm(range(0, len(items_df), batch_size), desc="Indexing items"):
        batch = items_df.iloc[start:start + batch_size].copy()
        docs = batch["item_text"].fillna("").astype(str).tolist()
        embeddings = embedding_model.encode(
            docs,
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()
        ids = batch["item_key"].tolist()
        metadatas = []
        for r in batch.itertuples(index=False):
            metadatas.append({
                "domain": str(r.domain),
                "parent_asin": str(r.parent_asin),
                "title": str(getattr(r, "title", ""))[:500],
                "category_key": str(getattr(r, "category_key", ""))[:500],
                "popularity_score": float(getattr(r, "popularity_score", 0.0)),
                "gateway_score": float(getattr(r, "gateway_score", 0.0)),
                "item_train_count": int(getattr(r, "item_train_count", 0)),
            })
        collection.add(ids=ids, documents=docs, embeddings=embeddings, metadatas=metadatas)

print("Final Chroma count:", collection.count())

Loading embedding model: BAAI/bge-small-en-v1.5


/Users/habeebullahagbaje/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Current Chroma count: 0 Expected: 50927
Building/rebuilding Chroma collection...


Indexing items:   0%|          | 0/199 [00:00<?, ?it/s]

## 5. User memory/query builders

These create a compact Personalized Memory from train only. No LLM is used here yet.

In [ ]:
train_items = train_df.merge(
    items_df[[
        "domain", "parent_asin", "item_key", "title", "categories", "main_category", "category_key", "item_text", "item_card"
    ]],
    on=["domain", "parent_asin", "item_key"],
    how="left",
)

USER_GROUPS = {uid: g.sort_values(["timestamp", "domain", "parent_asin"]) for uid, g in train_items.groupby("user_id")}
USER_SEEN = {uid: set(g["item_key"].astype(str)) for uid, g in train_df.groupby("user_id")}


def _safe_join(values, max_items=12):
    vals = [str(v).strip() for v in values if str(v).strip() and str(v).strip().lower() != "nan"]
    seen = []
    for v in vals:
        if v not in seen:
            seen.append(v)
    return "; ".join(seen[:max_items])


def build_user_memory(user_id, source_domain=None, max_reviews=60):
    """Build compact user memory/profile query from train history only."""
    if user_id not in USER_GROUPS:
        return None

    hist = USER_GROUPS[user_id].copy()
    if source_domain is not None:
        hist = hist[hist["domain"] == source_domain].copy()
    if hist.empty:
        return None

    mean_rating = hist["rating"].mean()
    rating_dist = hist["rating"].round().value_counts().sort_index().to_dict()

    positive = hist[hist["rating"] >= 4].sort_values(["rating", "timestamp"], ascending=[False, False]).head(8)
    negative = hist[hist["rating"] <= 2].sort_values(["rating", "timestamp"], ascending=[True, False]).head(5)
    recent_likes = hist[hist["rating"] >= 4].sort_values("timestamp", ascending=False).head(6)
    mid = hist[(hist["rating"] > 2) & (hist["rating"] < 4)].sort_values("timestamp", ascending=False).head(4)

    liked_titles = _safe_join(positive["title"].fillna("").tolist(), max_items=10)
    disliked_titles = _safe_join(negative["title"].fillna("").tolist(), max_items=6)
    recent_like_titles = _safe_join(recent_likes["title"].fillna("").tolist(), max_items=8)
    liked_categories = _safe_join(positive["category_key"].fillna("").tolist(), max_items=10)
    domains = _safe_join(hist["domain"].fillna("").tolist(), max_items=5)

    review_snips = []
    for r in pd.concat([positive.head(4), negative.head(2), mid.head(2)]).itertuples(index=False):
        text = str(getattr(r, "review_text", ""))[:300].replace("\n", " ")
        title = str(getattr(r, "title", ""))[:120]
        review_snips.append(f"- rating={getattr(r, 'rating', '')}; item={title}; review={text}")

    profile_query = f"""
User recommendation profile based on Amazon review history.
Domains seen: {domains}
Average rating: {mean_rating:.2f}; rating distribution: {rating_dist}
Liked titles/items: {liked_titles}
Recent liked items: {recent_like_titles}
Disliked or low-rated items: {disliked_titles}
Liked categories/themes: {liked_categories}
Representative review evidence:
{chr(10).join(review_snips)}
Recommend items that match the user's recurring preferences and avoid low-rated patterns.
""".strip()

    anchor_rows = pd.concat([recent_likes, positive]).drop_duplicates("item_key").head(8)
    anchor_queries = anchor_rows["item_text"].fillna("").astype(str).tolist()

    return {
        "user_id": user_id,
        "source_domain": source_domain,
        "n_train_reviews": int(len(hist)),
        "mean_rating": float(mean_rating),
        "rating_dist": rating_dist,
        "profile_query": profile_query,
        "anchor_queries": anchor_queries,
        "liked_categories": [c for c in positive["category_key"].dropna().astype(str).unique().tolist() if c][:10],
        "positive_item_keys": positive["item_key"].astype(str).tolist(),
        "negative_item_keys": negative["item_key"].astype(str).tolist(),
    }

example_user = internal_val_df["user_id"].iloc[0]
mem = build_user_memory(example_user)
print(mem["profile_query"][:1200])

## 6. Chroma retrieval + popularity/gateway/category candidate union

In [ ]:
def embed_query(texts):
    if isinstance(texts, str):
        texts = [texts]
    return embedding_model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()


def chroma_search_texts(query_texts, n_results=500, target_domain=None):
    """Search Chroma with one or more query texts; return combined item scores."""
    if isinstance(query_texts, str):
        query_texts = [query_texts]
    query_texts = [q for q in query_texts if isinstance(q, str) and q.strip()]
    if not query_texts:
        return pd.DataFrame(columns=["domain", "parent_asin", "item_key", "chroma_score"])

    where = {"domain": target_domain} if target_domain else None
    q_embs = embed_query(query_texts)
    res = collection.query(
        query_embeddings=q_embs,
        n_results=n_results,
        where=where,
        include=["metadatas", "distances", "documents"],
    )

    scores = defaultdict(lambda: {"domain": None, "parent_asin": None, "title": None, "chroma_score": -999.0})
    for ids, metas, dists in zip(res["ids"], res["metadatas"], res["distances"]):
        for item_id, meta, dist in zip(ids, metas, dists):
            score = 1.0 - float(dist)  # cosine distance -> similarity-ish score
            if score > scores[item_id]["chroma_score"]:
                scores[item_id] = {
                    "domain": str(meta.get("domain", "")),
                    "parent_asin": str(meta.get("parent_asin", "")),
                    "title": str(meta.get("title", "")),
                    "item_key": item_id,
                    "chroma_score": score,
                }
    out = pd.DataFrame(scores.values())
    if len(out):
        out = out.sort_values("chroma_score", ascending=False).reset_index(drop=True)
    return out


def get_popular_candidates(target_domain=None, top_n=300):
    df = items_df.copy()
    if target_domain:
        df = df[df["domain"] == target_domain]
    out = df.sort_values("popularity_score", ascending=False).head(top_n).copy()
    out["popular_score"] = out["popularity_score"]
    return out[["domain", "parent_asin", "item_key", "title", "popular_score"]]


def get_gateway_candidates(target_domain=None, top_n=300):
    df = items_df.copy()
    if target_domain:
        df = df[df["domain"] == target_domain]
    out = df.sort_values("gateway_score", ascending=False).head(top_n).copy()
    out["gateway_candidate_score"] = out["gateway_score"]
    return out[["domain", "parent_asin", "item_key", "title", "gateway_candidate_score"]]


def get_category_popular_candidates(user_memory, target_domain=None, top_n=300):
    cats = user_memory.get("liked_categories", []) if user_memory else []
    if not cats:
        return pd.DataFrame(columns=["domain", "parent_asin", "item_key", "title", "category_pop_score"])
    df = items_df.copy()
    if target_domain:
        df = df[df["domain"] == target_domain]
    df = df[df["category_key"].isin(cats)].copy()
    if df.empty:
        return pd.DataFrame(columns=["domain", "parent_asin", "item_key", "title", "category_pop_score"])
    out = df.sort_values("popularity_score", ascending=False).head(top_n).copy()
    out["category_pop_score"] = out["popularity_score"]
    return out[["domain", "parent_asin", "item_key", "title", "category_pop_score"]]


def minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    lo, hi = float(s.min()), float(s.max())
    if hi <= lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def union_candidate_sources(source_frames, seen_item_keys=None, top_n=1000):
    """Union candidate dataframes on item_key and combine source scores."""
    seen_item_keys = seen_item_keys or set()
    merged = None
    for name, df in source_frames.items():
        if df is None or df.empty:
            continue
        tmp = df.copy().drop_duplicates("item_key")
        tmp[f"source_{name}"] = 1
        keep = [c for c in tmp.columns if c in {"domain", "parent_asin", "item_key", "title"} or c.endswith("score") or c.startswith("source_")]
        tmp = tmp[keep]
        if merged is None:
            merged = tmp
        else:
            merged = merged.merge(tmp, on=["domain", "parent_asin", "item_key"], how="outer", suffixes=("", f"_{name}"))
            title_cols = [c for c in merged.columns if c == "title" or c.startswith("title_")]
            merged["title"] = merged[title_cols].bfill(axis=1).iloc[:, 0]
            merged = merged.drop(columns=[c for c in title_cols if c != "title"])

    if merged is None or merged.empty:
        return pd.DataFrame()

    merged = merged[~merged["item_key"].isin(seen_item_keys)].copy()

    score_cols = [c for c in merged.columns if c.endswith("score")]
    source_cols = [c for c in merged.columns if c.startswith("source_")]
    for c in score_cols + source_cols:
        merged[c] = merged[c].fillna(0.0)

    for c in score_cols:
        merged[c + "_norm"] = minmax(merged[c])

    merged["blend_score"] = 0.0
    weights = {
        "profile_score_norm": 0.32,
        "anchor_score_norm": 0.32,
        "popular_score_norm": 0.10,
        "gateway_candidate_score_norm": 0.10,
        "category_pop_score_norm": 0.16,
    }
    for col, w in weights.items():
        if col in merged.columns:
            merged["blend_score"] += w * merged[col]

    if source_cols:
        merged["n_sources"] = merged[source_cols].sum(axis=1)
        merged["blend_score"] += 0.03 * merged["n_sources"]
    else:
        merged["n_sources"] = 1

    return merged.sort_values("blend_score", ascending=False).head(top_n).reset_index(drop=True)

## 7. Main hybrid Chroma candidate generator

In [ ]:
def recommend_chroma_hybrid(
    user_id,
    target_domain=None,
    source_domain=None,
    top_n=FINAL_CANDIDATE_K,
    n_profile=N_PROFILE,
    n_anchor_total=N_ANCHOR_TOTAL,
    n_popular=N_POPULAR,
    n_gateway=N_GATEWAY,
    n_category_popular=N_CATEGORY_POPULAR,
):
    """
    High-recall candidate generator.

    target_domain: restrict recommended items to a domain, useful for eval or cross-domain.
    source_domain: restrict user memory to one domain, useful for simulated cross-domain.
    """
    user_memory = build_user_memory(user_id, source_domain=source_domain)
    if user_memory is None:
        sources = {
            "popular": get_popular_candidates(target_domain, n_popular),
            "gateway": get_gateway_candidates(target_domain, n_gateway),
        }
        return union_candidate_sources(sources, seen_item_keys=set(), top_n=top_n)

    seen = USER_SEEN.get(user_id, set())

    profile_df = chroma_search_texts(
        user_memory["profile_query"],
        n_results=n_profile,
        target_domain=target_domain,
    ).rename(columns={"chroma_score": "profile_score"})

    anchor_queries = user_memory.get("anchor_queries", [])[:8]
    per_anchor = max(50, int(math.ceil(n_anchor_total / max(1, len(anchor_queries))))) if anchor_queries else 0
    anchor_df = chroma_search_texts(
        anchor_queries,
        n_results=per_anchor,
        target_domain=target_domain,
    ).rename(columns={"chroma_score": "anchor_score"})
    if len(anchor_df) > n_anchor_total:
        anchor_df = anchor_df.head(n_anchor_total)

    sources = {
        "profile": profile_df,
        "anchor": anchor_df,
        "popular": get_popular_candidates(target_domain, n_popular),
        "gateway": get_gateway_candidates(target_domain, n_gateway),
        "category_pop": get_category_popular_candidates(user_memory, target_domain, n_category_popular),
    }
    return union_candidate_sources(sources, seen_item_keys=seen, top_n=top_n)

# Smoke test
user_id = internal_val_df["user_id"].iloc[0]
cands = recommend_chroma_hybrid(user_id, target_domain=None, top_n=20)
cands.head(10)

## 8. Per-user evaluation helpers

In [ ]:
def rank_metrics(recommended_keys, truth_keys, k=10):
    recommended_at_k = recommended_keys[:k]
    hits = [1 if item in truth_keys else 0 for item in recommended_at_k]
    hit_at_k = 1.0 if any(hits) else 0.0
    recall_at_k = sum(hits) / max(1, len(truth_keys))

    dcg = 0.0
    for rank_idx, hit in enumerate(hits, start=1):
        if hit:
            dcg += 1.0 / math.log2(rank_idx + 1)

    ideal_hits = min(len(truth_keys), k)
    idcg = sum(1.0 / math.log2(rank_idx + 1) for rank_idx in range(1, ideal_hits + 1))
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0

    return {
        f"hit_at_{k}": hit_at_k,
        f"recall_at_{k}": recall_at_k,
        f"ndcg_at_{k}": ndcg_at_k,
        f"n_hits_at_{k}": int(sum(hits)),
    }


def evaluate_per_user_ground_truth(
    eval_df,
    candidate_fn,
    *,
    group_by_domain=False,
    candidate_k=1000,
    eval_ks=(10, 30, 100, 500, 1000),
    max_users=None,
    random_state=42,
    verbose=True,
):
    df = eval_df[["user_id", "domain", "parent_asin", "item_key"]].dropna().drop_duplicates().copy()
    group_cols = ["user_id", "domain"] if group_by_domain else ["user_id"]
    groups = list(df.groupby(group_cols))

    if max_users is not None and len(groups) > max_users:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(len(groups), size=max_users, replace=False)
        groups = [groups[i] for i in idx]

    rows = []
    iterator = tqdm(groups, desc="eval users") if verbose else groups

    for group_key, truth_group in iterator:
        if group_by_domain:
            user_id, target_domain = group_key
        else:
            user_id = group_key[0] if isinstance(group_key, tuple) else group_key
            target_domain = None

        truth_keys = set(truth_group["item_key"].astype(str))
        if not truth_keys:
            continue

        candidates = candidate_fn(user_id=user_id, target_domain=target_domain, top_n=candidate_k)
        if candidates is None or candidates.empty:
            recommended_keys = []
        else:
            candidates = candidates.drop_duplicates("item_key").head(candidate_k)
            recommended_keys = candidates["item_key"].astype(str).tolist()

        row = {
            "user_id": user_id,
            "target_domain": target_domain if target_domain else "ALL",
            "n_truth_items": len(truth_keys),
            "n_candidates": len(recommended_keys),
        }
        for k in eval_ks:
            row.update(rank_metrics(recommended_keys, truth_keys, k=k))
        rows.append(row)

    results = pd.DataFrame(rows)
    metric_cols = [c for c in results.columns if c.startswith("hit_at_") or c.startswith("recall_at_") or c.startswith("ndcg_at_")]
    summary = results[metric_cols].mean().to_dict() if len(results) else {}
    return results, summary

## 9. Evaluate Chroma hybrid retrieval

Start with `max_users=200` for speed. If promising, increase to 1000 or `None`.

Use `group_by_domain=False` first to match your latest user-level evaluation. Then test `group_by_domain=True` for stricter domain-specific eval.

In [ ]:
def chroma_hybrid_wrapper(user_id, target_domain=None, top_n=1000):
    return recommend_chroma_hybrid(user_id=user_id, target_domain=target_domain, top_n=top_n)

retrieval_results_200, retrieval_summary_200 = evaluate_per_user_ground_truth(
    internal_val_df,
    chroma_hybrid_wrapper,
    group_by_domain=False,
    candidate_k=1000,
    eval_ks=EVAL_KS,
    max_users=200,
    random_state=RANDOM_STATE,
)

retrieval_summary_200

In [ ]:
# Larger run. Increase max_users or set to None when ready.
# retrieval_results_all, retrieval_summary_all = evaluate_per_user_ground_truth(
#     internal_val_df,
#     chroma_hybrid_wrapper,
#     group_by_domain=False,
#     candidate_k=1000,
#     eval_ks=EVAL_KS,
#     max_users=1000,
#     random_state=RANDOM_STATE,
# )
# retrieval_summary_all

## 10. Source ablations

This helps identify whether profile retrieval, anchor retrieval, popularity, gateway, or category popularity is actually helping.

In [ ]:
def recommend_source_ablation(user_id, target_domain=None, top_n=1000, mode="profile_anchor"):
    user_memory = build_user_memory(user_id)
    seen = USER_SEEN.get(user_id, set())
    sources = {}

    if user_memory is None:
        sources["popular"] = get_popular_candidates(target_domain, top_n)
        return union_candidate_sources(sources, seen, top_n)

    if mode in ["profile", "profile_anchor", "all"]:
        sources["profile"] = chroma_search_texts(
            user_memory["profile_query"], n_results=top_n, target_domain=target_domain
        ).rename(columns={"chroma_score": "profile_score"})

    if mode in ["anchor", "profile_anchor", "all"]:
        anchor_queries = user_memory.get("anchor_queries", [])[:8]
        per_anchor = max(50, int(math.ceil(top_n / max(1, len(anchor_queries))))) if anchor_queries else 0
        sources["anchor"] = chroma_search_texts(
            anchor_queries, n_results=per_anchor, target_domain=target_domain
        ).rename(columns={"chroma_score": "anchor_score"}).head(top_n)

    if mode in ["popular", "all"]:
        sources["popular"] = get_popular_candidates(target_domain, top_n)

    if mode in ["gateway", "all"]:
        sources["gateway"] = get_gateway_candidates(target_domain, top_n)

    if mode in ["category_pop", "all"]:
        sources["category_pop"] = get_category_popular_candidates(user_memory, target_domain, top_n)

    return union_candidate_sources(sources, seen_item_keys=seen, top_n=top_n)

ablation_summaries = {}
for mode in ["profile", "anchor", "profile_anchor", "popular", "gateway", "category_pop", "all"]:
    print("\nRunning mode:", mode)
    def fn(user_id, target_domain=None, top_n=1000, mode=mode):
        return recommend_source_ablation(user_id, target_domain, top_n, mode=mode)
    _, summ = evaluate_per_user_ground_truth(
        internal_val_df,
        fn,
        group_by_domain=False,
        candidate_k=1000,
        eval_ks=EVAL_KS,
        max_users=200,
        random_state=RANDOM_STATE,
        verbose=False,
    )
    ablation_summaries[mode] = summ

pd.DataFrame(ablation_summaries).T

## 11. Optional cross-encoder rerank

This tests whether local pairwise reranking can push true items higher. Start with a small user sample because this scores many `(user_profile, item)` pairs.

Recommended use:

```text
retrieve top 500 or 1000 → cross-encoder rerank → evaluate top 10/30/100
```

In [ ]:
LOAD_CROSS_ENCODER = False

cross_encoder = None
if LOAD_CROSS_ENCODER:
    print("Loading cross encoder:", CROSS_ENCODER_MODEL_NAME)
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME)
    print("Loaded cross encoder.")

In [ ]:
def make_cross_encoder_query(user_id, target_domain=None, source_domain=None):
    mem = build_user_memory(user_id, source_domain=source_domain)
    if mem is None:
        base = "Recommend broadly appealing high-quality Amazon items."
    else:
        base = mem["profile_query"]
    if target_domain:
        base += f"\nTarget recommendation domain: {target_domain}."
    return base[:2500]


def attach_item_cards(candidates):
    if candidates is None or candidates.empty:
        return pd.DataFrame()
    return candidates.merge(
        items_df[["domain", "parent_asin", "item_key", "item_card", "popularity_score", "gateway_score", "item_train_count"]],
        on=["domain", "parent_asin", "item_key"],
        how="left",
    )


def rerank_with_cross_encoder(user_id, candidates, target_domain=None, top_n=100, batch_size=32):
    if cross_encoder is None:
        raise RuntimeError("Set LOAD_CROSS_ENCODER=True and run the loading cell first.")
    if candidates is None or candidates.empty:
        return candidates

    df = attach_item_cards(candidates).drop_duplicates("item_key").copy()
    query = make_cross_encoder_query(user_id, target_domain=target_domain)
    pairs = [(query, str(card)[:1600]) for card in df["item_card"].fillna("").tolist()]
    scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    df["cross_encoder_score"] = scores

    df["ce_norm"] = minmax(df["cross_encoder_score"])
    df["blend_norm"] = minmax(df.get("blend_score", pd.Series(np.zeros(len(df)), index=df.index)))
    df["rerank_score"] = 0.75 * df["ce_norm"] + 0.25 * df["blend_norm"]
    return df.sort_values("rerank_score", ascending=False).head(top_n).reset_index(drop=True)


def chroma_plus_cross_encoder_wrapper(user_id, target_domain=None, top_n=100):
    base = recommend_chroma_hybrid(user_id=user_id, target_domain=target_domain, top_n=1000)
    return rerank_with_cross_encoder(user_id, base.head(500), target_domain=target_domain, top_n=top_n)

print("Cross-encoder functions ready. Set LOAD_CROSS_ENCODER=True above to run them.")

In [ ]:
# Example cross-encoder evaluation.
# First set LOAD_CROSS_ENCODER=True in the previous cell and rerun the loading cell.

# ce_results_50, ce_summary_50 = evaluate_per_user_ground_truth(
#     internal_val_df,
#     chroma_plus_cross_encoder_wrapper,
#     group_by_domain=False,
#     candidate_k=100,
#     eval_ks=(10, 30, 100),
#     max_users=50,
#     random_state=RANDOM_STATE,
# )
# ce_summary_50

## 12. Save run outputs

In [ ]:
OUT_DIR = Path("data/processed/task_b_chroma_experiments_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    retrieval_results_200.to_parquet(OUT_DIR / "retrieval_results_200.parquet", index=False)
    with open(OUT_DIR / "retrieval_summary_200.json", "w", encoding="utf-8") as f:
        json.dump(retrieval_summary_200, f, indent=2)
    print("Saved retrieval summary/results.")
except NameError:
    print("No retrieval_results_200 found yet.")

try:
    ablation_table = pd.DataFrame(ablation_summaries).T
    ablation_table.to_csv(OUT_DIR / "source_ablation_200.csv")
    print("Saved ablation table.")
except NameError:
    print("No ablation_summaries found yet.")

print("Output dir:", OUT_DIR)

## Notes for interpretation

Compare the Chroma hybrid retrieval to your previous benchmark:

```text
previous best user-level all-val:
hit@10 ≈ 0.10
hit@500 ≈ 0.445
```

The first target is not Hit@10. The first target is better candidate coverage:

```text
hit@500 > 0.445
hit@1000 ideally much higher
```

If Hit@1000 improves but Hit@10 does not, retrieval is improving and reranking is the next bottleneck.

If Hit@1000 is still weak, add more retrieval sources next:

```text
1. item-to-item collaborative filtering
2. local embedding anchor retrieval with more anchors
3. category-aware item-to-item retrieval
4. broader top-2000 retrieval before reranking
```
